# Data Preprocessing

This notebook prepares the obesity dataset for machine learning model development.

The preprocessing workflow converts raw data into a machine-learning-ready format by:

- Separating input features and the target variable.
- Splitting data into training, validation, and testing sets.
- Handling missing values.
- Scaling numerical features.
- Encoding categorical features.
- Combining all transformations into a single preprocessing pipeline.

This notebook demonstrates each preprocessing step, while `src/preprocessing.py` provides the reusable implementation used by the machine learning models.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

import warnings
warnings.filterwarnings("ignore")

## Dataset Loading

The raw obesity dataset is loaded from the project data directory.

The dataset contains lifestyle, physical, and demographic attributes that are used to predict the obesity category (`NObeyesdad`).


In [2]:
DATA_PATH = "../data/raw/obesity.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (20758, 18)
<class 'pandas.DataFrame'>
RangeIndex: 20758 entries, 0 to 20757
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              20758 non-null  int64  
 1   Gender                          20758 non-null  str    
 2   Age                             20758 non-null  float64
 3   Height                          20758 non-null  float64
 4   Weight                          20758 non-null  float64
 5   family_history_with_overweight  20758 non-null  str    
 6   FAVC                            20758 non-null  str    
 7   FCVC                            20758 non-null  float64
 8   NCP                             20758 non-null  float64
 9   CAEC                            20758 non-null  str    
 10  SMOKE                           20758 non-null  str    
 11  CH2O                            20758 non-null  float64
 12  SCC             

## Feature and Target Separation

The dataset contains:

- Input features used for prediction
- A target variable (`NObeyesdad`) representing obesity categories
- An identifier column (`id`)

The identifier column does not contain meaningful predictive information, therefore it is removed before model development.

In [4]:
target_column = "NObeyesdad"
identifier_column = "id"

X = df.drop(
    columns=[
        identifier_column,
        target_column
    ]
)

y = df[target_column]

## Feature Classification

Different types of features require different preprocessing techniques.

The features are divided into:

1. Numerical Features
   - Continuous numerical values
   - Processed using imputation and scaling

2. Ordinal Features
   - Categorical values with a meaningful order
   - Processed using ordinal encoding

3. Nominal Features
   - Categorical values without an inherent order
   - Processed using one-hot encoding

In [5]:
numerical_features = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE"
]

numerical_features

['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

In [6]:
ordinal_features = [
    "CAEC",
    "CALC"
]

ordinal_features

['CAEC', 'CALC']

In [7]:
nominal_features = [
    "Gender",
    "family_history_with_overweight",
    "FAVC",
    "SMOKE",
    "SCC",
    "MTRANS"
]

nominal_features

['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC', 'MTRANS']

In [8]:
all_features = (
    numerical_features +
    ordinal_features +
    nominal_features
)

print("Total Features:", len(all_features))
print("Dataset Features:", X.shape[1])

print(
    "All features covered:",
    set(all_features) == set(X.columns)
)

Total Features: 16
Dataset Features: 16
All features covered: True


## Train, Validation, and Test Split

The dataset is divided into three subsets:

- Training set:
  Used for learning model patterns

- Validation set:
  Used for model comparison and tuning

- Testing set:
  Used for final performance evaluation

Stratified splitting is applied to maintain the same target class distribution across all datasets.

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [10]:
print("Training set:", X_train.shape)
print("Validation set:", X_valid.shape)
print("Testing set:", X_test.shape)

Training set: (14530, 16)
Validation set: (3114, 16)
Testing set: (3114, 16)


## Numerical Feature Preprocessing Pipeline

Numerical features are processed using:

1. Simple Imputer
   - Handles missing numerical values using the median value

2. Standard Scaler
   - Transforms features into a similar scale
   - Prevents features with larger ranges from dominating model learning

In [11]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

## Ordinal Category Ordering

Ordinal features contain categories with a natural order.

The category order must be manually defined before encoding because the model should understand the relationship between categories.

Example:

CAEC:
no < Sometimes < Frequently < Always

CALC:
no < Sometimes < Frequently

In [12]:
ordinal_categories = [
    ["no", "Sometimes", "Frequently", "Always"],
    ["no", "Sometimes", "Frequently"]
]

## Ordinal Feature Preprocessing Pipeline

Ordinal features contain categories where the order has meaning.

Example:

CAEC:
No < Sometimes < Frequently < Always

These features are processed using Ordinal Encoding so that the category order is preserved.

In [13]:
ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)

## Nominal Feature Preprocessing Pipeline

Nominal features contain categories without any natural order.

Examples:

- Gender
- Transportation method

These features are processed using One-Hot Encoding to convert categories into numerical representations.

In [14]:
nominal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

## Combined Preprocessing Pipeline

The three preprocessing pipelines are combined using a `ColumnTransformer`.

It applies a different transformation to each feature group:

- **Numerical features:** Fill missing values using the median and standardize numerical values.
- **Ordinal features:** Fill missing categories and encode responses while preserving their meaningful order.
- **Nominal features:** Fill missing categories and convert responses into separate numerical columns using one-hot encoding.

The combined pipeline prepares all 16 original predictive features consistently.

### Why Do We Also Have `src/preprocessing.py`?

This notebook demonstrates how preprocessing works by building each pipeline step by step.

The project also contains a reusable Python module:

`src/preprocessing.py`

It defines the predictive feature groups, category ordering, and a function called `build_preprocessor()` that constructs the same preprocessing pipeline.

The difference is:

| Notebook 3 | `src/preprocessing.py` |
|---|---|
| Explains and demonstrates preprocessing step by step | Stores preprocessing logic in a reusable Python function |
| Displays transformed data and results for understanding | Creates a preprocessor that other project components can import |
| Focuses on learning and verifying the process | Helps maintain consistent preprocessing during model training and prediction |

Both implementations should follow the same preprocessing rules.

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        ),
        (
            "nominal",
            nominal_pipeline,
            nominal_features
        )
    ]
)

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)

X_valid_processed = preprocessor.transform(X_valid)

X_test_processed = preprocessor.transform(X_test)

## Applying the Preprocessing Pipeline

The preprocessing pipeline is fitted only using the training data.

The learned transformations are then applied to validation and testing datasets.

This prevents data leakage and ensures that evaluation data is processed using the same transformations learned from the training data.

In [17]:
print("Training data shape:", X_train_processed.shape)
print("Validation data shape:", X_valid_processed.shape)
print("Testing data shape:", X_test_processed.shape)

Training data shape: (14530, 25)
Validation data shape: (3114, 25)
Testing data shape: (3114, 25)


## Preprocessing Output Validation

The transformed datasets are checked to confirm that:

- The transformation process completed successfully
- The number of generated features is correct
- Training, validation, and testing data have consistent feature structures

In [18]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

feature_names[:20]

Total processed features: 25


array(['numerical__Age', 'numerical__Height', 'numerical__Weight',
       'numerical__FCVC', 'numerical__NCP', 'numerical__CH2O',
       'numerical__FAF', 'numerical__TUE', 'ordinal__CAEC',
       'ordinal__CALC', 'nominal__Gender_Female', 'nominal__Gender_Male',
       'nominal__family_history_with_overweight_no',
       'nominal__family_history_with_overweight_yes', 'nominal__FAVC_no',
       'nominal__FAVC_yes', 'nominal__SMOKE_no', 'nominal__SMOKE_yes',
       'nominal__SCC_no', 'nominal__SCC_yes'], dtype=object)

In [19]:
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=feature_names
)

X_train_processed_df.head()

,numerical__Age,numerical__Height,numerical__Weight,numerical__FCVC,numerical__NCP,numerical__CH2O,numerical__FAF,numerical__TUE,ordinal__CAEC,ordinal__CALC,...,nominal__FAVC_yes,nominal__SMOKE_no,nominal__SMOKE_yes,nominal__SCC_no,nominal__SCC_yes,nominal__MTRANS_Automobile,nominal__MTRANS_Bike,nominal__MTRANS_Motorbike,nominal__MTRANS_Public_Transportation,nominal__MTRANS_Walking
0,0.385914,-0.881557,0.910609,1.038204,0.343418,1.236961,-1.167175,-0.820903,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.373562,0.180375,0.948677,1.038204,0.343418,1.161405,-1.122673,-0.771579,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.408144,0.638226,1.215992,0.976776,0.343418,0.645981,-0.024391,-1.026618,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.213334,0.889680,1.006426,-2.514706,0.343418,-0.042607,-0.764289,-1.011249,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.031084,-0.799056,-0.146914,1.038204,-2.476780,1.592657,0.029060,-1.026618,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [20]:
print("Processed training shape:")
print(X_train_processed_df.shape)

print("\nMissing values:")
print(X_train_processed_df.isnull().sum().sum())

Processed training shape:
(14530, 25)

Missing values:
0


## Shared Preprocessor Verification

We have built a preprocessing pipeline step by step in this notebook.

Now we check whether it produces the same results as the reusable `build_preprocessor()` function in `src/preprocessing.py`.

The comparison checks:

- Whether both implementations use the same 16 predictive features.
- Whether both produce the same transformed feature names and order.
- Whether both produce the same numerical values for the training, validation, and testing data.

Both preprocessors are fitted only on the training data. Validation and testing data are transformed using the fitted preprocessing rules.

If the results match, it confirms that the notebook's demonstrated preprocessing process is consistent with the reusable implementation.

In [21]:
from pathlib import Path
import sys


# Locate the project root
PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src" / "preprocessing.py").exists()
    ),
    None,
)

assert PROJECT_ROOT is not None, "Project root not found."


# Allow Python to import the shared preprocessing module
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Import the reusable preprocessing implementation
from src.preprocessing import (
    PREDICTIVE_FEATURES,
    build_preprocessor,
)


# Build the reusable preprocessing pipeline
shared_preprocessor = build_preprocessor()


# Fit only on training data
shared_train_processed = shared_preprocessor.fit_transform(
    X_train
)


# Apply the fitted transformations to validation and test data
shared_valid_processed = shared_preprocessor.transform(
    X_valid
)

shared_test_processed = shared_preprocessor.transform(
    X_test
)


# Check original predictive features
assert all_features == PREDICTIVE_FEATURES


# Check transformed feature names and order
assert np.array_equal(
    feature_names,
    shared_preprocessor.get_feature_names_out(),
)


# Compare the transformed numerical values
assert np.allclose(
    X_train_processed,
    shared_train_processed,
)

assert np.allclose(
    X_valid_processed,
    shared_valid_processed,
)

assert np.allclose(
    X_test_processed,
    shared_test_processed,
)


print("Shared preprocessing verification passed.")

print("Original predictive features:", len(PREDICTIVE_FEATURES))

print("Transformed features:", len(feature_names))

print("Training values match:", True)

print("Validation values match:", True)

print("Testing values match:", True)

Shared preprocessing verification passed.
Original predictive features: 16
Transformed features: 25
Training values match: True
Validation values match: True
Testing values match: True


## Preprocessing Completed

The dataset has been prepared for machine learning using the appropriate transformations for each feature type.

The preprocessing process includes:

- Separating the 16 predictive features and target variable.
- Creating stratified training, validation, and testing sets.
- Handling missing values.
- Scaling numerical features.
- Encoding ordinal and nominal features.
- Fitting preprocessing only on training data to prevent data leakage.

The original 16 predictors are transformed into 25 numerical features.

The shared preprocessing verification confirms that the notebook's implementation produces the same results as `build_preprocessor()` in `src/preprocessing.py`.

The project can therefore reuse the same preprocessing logic when building and applying classification models.